# Symbolic Reasoning Agent
Aniruddha Karmokar (P26CS0001)

Natural language → logic formulas → forward chaining / DPLL / causal model → True or False.

Set `TEST_FILE` below to a `.jsonl`, `.json` or `.csv` file with `id`, `context`, `statement` (labels optional) and run all cells.
The public data in `DATA_DIR` is optional.

In [1]:
import re, json, csv, time, itertools
from collections import defaultdict, Counter
from pathlib import Path

DATA_DIR = Path('data')
TEST_FILE = None


def truth(v):
    return str(v).strip().lower() in ('true', 'yes', '1', 'entailment', 'entailed')


def load(path):
    path = Path(path)
    text = path.read_text(encoding='utf-8-sig')
    if path.suffix == '.csv':
        raw = list(csv.DictReader(text.splitlines()))
    elif path.suffix == '.json':
        raw = json.loads(text)
    else:
        raw = [json.loads(line) for line in text.splitlines() if line.strip()]
    rows = []
    for i, r in enumerate(raw):
        context = next((r[k] for k in ('context', 'premises', 'theory', 'given_info') if k in r), '')
        statement = next((r[k] for k in ('statement', 'query', 'question', 'conclusion', 'hypothesis') if k in r), '')
        row = {**r, 'id': r.get('id', i), 'statement': str(statement),
               'context': '\n'.join(map(str, context)) if isinstance(context, list) else str(context),
               'dataset': r.get('dataset', path.stem), 'split': r.get('split', 'all')}
        row.pop('label', None)
        if r.get('label') not in (None, ''):
            row['label'] = truth(r['label'])
        rows.append(row)
    return rows


rows = [r for p in sorted(DATA_DIR.glob('*.jsonl')) for r in load(p)] if DATA_DIR.is_dir() else []
print(f'{len(rows)} public rows loaded')
for k, v in sorted(Counter((r['dataset'], r['split']) for r in rows).items()):
    print(' ', k, v)

5029 public rows loaded
  ('CounterBench', 'public') 1200
  ('FOLIO', 'train') 1001
  ('FOLIO', 'validation') 203
  ('RuleTaker', 'ood_test') 375
  ('RuleTaker', 'test') 250
  ('RuleTaker', 'train') 1000
  ('RuleTaker-standard', 'synthetic') 1000


## 1. Text normalisation

In [2]:
ART = {'a', 'an', 'the'}
BE = {'is', 'are', 'am', 'was', 'were', 'be', 'been', 'being'}
DO = {'do', 'does', 'did'}
FINITE = {'is', 'are', 'am', 'was', 'were', 'has', 'have', 'had', 'does', 'do', 'did',
          'can', 'could', 'will', 'would', 'should', 'must', 'may', 'might', 'not'}
PLURAL_VERB = {'are', 'were', 'have', 'do'}
PRON = {'he', 'she', 'they', 'it', 'him', 'her', 'them'}
VARS = {'someone', 'something', 'somebody', 'anyone', 'anything', 'anybody',
        'everyone', 'everything', 'everybody'}
GENERIC = VARS | {'person', 'one', 'thing', 'those'}
IRREG = {'people': 'person', 'men': 'man', 'women': 'woman', 'children': 'child',
         'has': 'have', 'goes': 'go', 'mice': 'mouse'}


def lemma(w):
    if w in IRREG:
        return IRREG[w]
    if len(w) > 4 and w.endswith('ies'):
        return w[:-3] + 'y'
    if len(w) > 4 and w.endswith(('ches', 'shes', 'sses', 'xes')):
        return w[:-2]
    if len(w) > 3 and w.endswith('s') and not w.endswith(('ss', 'us', 'is')):
        return w[:-1]
    return w


def stem(w):
    w = lemma(w)
    if len(w) > 5 and w.endswith('ing'):
        w = w[:-3]
    elif len(w) > 4 and w.endswith('ed'):
        w = w[:-2]
    else:
        return w[:-1] if len(w) > 3 and w.endswith('e') else w
    if len(w) > 2 and w[-1] == w[-2] and w[-1] not in 'lsz':
        w = w[:-1]
    return w[:-1] if len(w) > 3 and w.endswith('e') else w


def clean(t):
    t = t.lower().replace('’', "'")
    t = re.sub(r"\bcan't\b|\bcannot\b", 'can not', t)
    t = re.sub(r"\bwon't\b", 'will not', t)
    t = re.sub(r"n't\b", ' not', t)
    t = re.sub(r"'s\b", '', t)
    t = re.sub(r"[^a-z0-9' ,\-]", ' ', t)
    return re.sub(r'\s+', ' ', t).strip(' ,')


def pred(text):
    return ' '.join(w if w == 'X' else stem(w) for w in text.split() if w not in ART | BE | DO | {'not', 'never'})


def atom(p, t):
    return ('atom', p, t)


def neg(f):
    return f[1] if f[0] == 'not' else ('not', f)


def has_var(f):
    if f[0] == 'atom':
        return f[2] == 'X' or 'X' in f[1].split()
    return any(has_var(g) for g in f[1:])


print(pred('is not dancing gracefully'), '|', pred('dances gracefully'), '|', pred('are students'))

danc gracefully | danc gracefully | student


## 2. Sentence parser

In [3]:
class Parser:
    def __init__(self):
        self.consts = []
        self.objects = set()
        self.rule = False

    def add_const(self, c):
        if c not in self.consts:
            self.consts.append(c)
            self.consts.sort(key=len, reverse=True)

    def split(self, text):
        for c in self.consts:
            for pre in (c + ' ', 'the ' + c + ' '):
                if text.startswith(pre):
                    return c, text[len(pre):]
        w = text.split()
        for i in range(1, len(w)):
            head = [x for x in w[:i] if x not in ART]
            if head and (w[i] in FINITE or lemma(w[i]) != w[i]):
                return ' '.join(head), ' '.join(w[i:])
        head = [x for x in w if x not in ART][:1]
        if not head or len(w) < 2:
            raise ValueError(text)
        return head[0], ' '.join(w[w.index(head[0]) + 1:])

    def literal(self, text, t):
        if self.rule:
            text = ' '.join('X' if w in VARS | {'it', 'them'} else w for w in text.split())
        p = pred(text)
        if not p:
            raise ValueError(text)
        m = re.search(r'\bthe ((?:\S+ )?\S+)$', text)
        if m and 'X' not in m[1]:
            self.objects.add(m[1])
        return neg(atom(p, t)) if re.search(r'\b(not|never)\b', text) else atom(p, t)

    def vp(self, text, t):
        text = text.strip(' ,')
        w = text.split()
        if w and w[0] in PRON | VARS:
            text = ' '.join(w[1:])
        elif t != 'X' and (text.startswith('the ') or any(text.startswith(c + ' ') for c in self.consts)):
            t, text = self.split(text)
        m = re.match(r'^(.+?),? and (?:he|she|it|they) (.+)$', text)
        if m:
            return ('and', self.vp(m[1], t), self.vp(m[2], t))
        m = re.match(r'^(?:(?:is|are|was|were) )?neither (.+?) nor (.+)$', text)
        if m:
            return ('and', neg(self.vp(m[1], t)), neg(self.vp(m[2], t)))
        either = re.match(r'^(?:(?:is|are|was|were) )?either ', text)
        if either:
            text = text[either.end():]
        if ' or ' in text:
            both = re.search(r',? or both$', text)
            if both:
                text = text[:both.start()]
            parts = [self.vp(p, t) for p in re.split(r',? or |, ', text) if p.strip()]
            if either and not both and len(parts) == 2:
                return ('xor',) + tuple(parts)
            return ('or',) + tuple(parts)
        text = re.sub(r'^(?:(?:is|are|was|were) )?both ', '', text)
        if ' and ' in text or ', ' in text:
            return ('and',) + tuple(self.vp(p, t) for p in re.split(r',? and |, ', text) if p.strip())
        m = re.match(r'^(.+?) (?:who|which) (.+)$', text)
        if m:
            return ('and', self.vp(m[1], t), self.vp(m[2], t))
        return self.literal(text, t)

    def np_split(self, text):
        w = text.split()
        rel = next((i for i, x in enumerate(w) if x in ('who', 'that', 'which')), None)
        start = 1 if rel is None else rel + 2
        for i in range(start, len(w)):
            if w[i] in FINITE and w[i - 1] not in ('and', 'or'):
                return ' '.join(w[:i]), ' '.join(w[i:])
        for i in range(start, len(w)):
            if lemma(w[i]) != w[i]:
                return ' '.join(w[:i]), ' '.join(w[i:])
        return w[0], ' '.join(w[1:])

    def np_cond(self, np):
        np = ' '.join(x for x in np.split() if x not in ART)
        m = re.match(r'^(.*?) (?:who|that|which) (.+)$', np)
        head, rel = (m[1], m[2]) if m else (np, None)
        head = re.sub(r'(?:^| )(?:things?|people|persons?|ones?)$', '', head)
        conds = [self.literal(h, 'X') for h in re.split(r',? and |, ', head) if pred(h) and h.strip() not in GENERIC]
        return conds + ([self.vp(rel, 'X')] if rel else [])

    def quantified(self, q, np, vp):
        self.rule = True
        conds = self.np_cond(np)
        body = self.vp(vp, 'X')
        if q in ('no', 'none of', 'none of the'):
            body = neg(body)
        if q in ('some', 'not all') or q.startswith('there'):
            return ('exists', ('and',) + tuple(conds) + (neg(body) if q == 'not all' else body,))
        if not conds:
            return body
        return ('imp', conds[0] if len(conds) == 1 else ('and',) + tuple(conds), body)

    def clause(self, text, t=None):
        if text.startswith('it is not true that '):
            f, s = self.clause(text[20:], t)
            return neg(f), s
        w = text.split()
        if w[0] in VARS or (t == 'X' and w[0] in PRON):
            self.rule = True
            return self.vp(' '.join(w[1:]), 'X'), 'X'
        if w[0] in ('a', 'an'):
            self.rule = True
            np, vp = self.np_split(' '.join(w[1:]))
            return ('and',) + tuple(self.np_cond(np)) + (self.vp(vp, 'X'),), 'X'
        if t and w[0] in PRON:
            return self.vp(' '.join(w[1:]), t), t
        s, rest = self.split(text)
        self.add_const(s)
        return self.vp(rest, s), s

    def conditional(self, a, b):
        self.rule = bool(re.search(r'\b(?:' + '|'.join(VARS) + r')\b', a))
        fa, t = self.clause(a)
        fb, _ = self.clause(b, 'X' if has_var(fa) else t)
        return ('imp', fa, fb)

    def sentence(self, s):
        self.rule = False
        t = re.sub(r'^in [^,]+, ', '', clean(s))
        m = re.match(r'^(.+?),? if and only if (.+)$', t)
        if m:
            fa, s1 = self.clause(m[1])
            return ('iff', fa, self.clause(m[2], s1)[0])
        conn = r'(?:then|we can say|it follows that|it is evident that|we can conclude(?: that)?)'
        start = r'^(?:if|when|whenever|assuming|given that|based on|provided that)'
        m = (re.match(start + r' (.+?),? ' + conn + r' (.+)$', t) or re.match(start + r' (.+?), (.+)$', t)
             or re.match(start + r' (.+?) (?<!and )((?:they|it|he|she) .+)$', t))
        if m:
            return self.conditional(m[1], m[2])
        m = re.match(r'^(.+?),? only if (.+)$', t)
        if m:
            return self.conditional(m[1], m[2])
        m = re.match(r'^(.+?),? if (.+)$', t)
        if m:
            fb, s1 = self.clause(m[2])
            return ('imp', fb, self.clause(m[1], s1)[0])
        m = re.match(r'^(all|every|each|any|no|some|not all|none of the|none of|there (?:is|are|exists?)(?: an?| some| at least one)?) (.+)$', t)
        if m:
            return self.quantified(m[1], *self.np_split(m[2]))
        w = t.split()
        if w[0] in VARS:
            if len(w) > 1 and w[1] in ('who', 'that'):
                return self.quantified('all', *self.np_split(t))
            self.rule = True
            return self.vp(' '.join(w[1:]), 'X')
        if w[0] in ('a', 'an', 'people', 'those'):
            return self.quantified('all', *self.np_split(' '.join(w[1:]) if w[0] in ('a', 'an') else t))
        np, vp = self.np_split(t)
        v = vp.split()[0] if vp else ''
        plural = lemma(np.split()[-1]) != np.split()[-1] and w[0] not in ART and np not in self.consts
        if plural and (v in PLURAL_VERB or (v not in FINITE and lemma(v) == v and s.split()[1].islower())):
            return self.quantified('all', np, vp)
        return self.clause(t)[0]


def sentences(context):
    if isinstance(context, list):
        return [str(s) for s in context if str(s).strip()]
    lines = [s.strip() for s in str(context).split('\n') if s.strip()]
    if len(lines) == 1:
        lines = re.split(r'(?<=[.!?])\s+(?=[A-Z])', lines[0])
    return [s for s in lines if s.lower().rstrip(':') not in ('facts', 'rules')]


def show(f):
    if f[0] == 'atom':
        return f'{f[1]}({f[2]})'
    if f[0] == 'not':
        return '¬' + show(f[1])
    if f[0] == 'exists':
        return '∃X ' + show(f[1])
    op = {'and': ' ∧ ', 'or': ' ∨ ', 'xor': ' ⊕ ', 'imp': ' → ', 'iff': ' ↔ '}[f[0]]
    return '(' + op.join(show(g) for g in f[1:]) + ')'


p = Parser()
for s in ['Bob is young.', 'If someone is smart and kind, then they are successful.',
          'All students who play chess are clever.', 'Some birds can not fly.',
          'Rina is either a student or a teacher.', 'Vacuums suck.', 'Red, round things are big.',
          'If something likes the cat and it is not red then the cat chases it.']:
    print(f'{s:70s} {show(p.sentence(s))}')

Bob is young.                                                          young(bob)
If someone is smart and kind, then they are successful.                ((smart(X) ∧ kind(X)) → successful(X))
All students who play chess are clever.                                ((student(X) ∧ play chess(X)) → clever(X))
Some birds can not fly.                                                ∃X (bird(X) ∧ ¬can fly(X))
Rina is either a student or a teacher.                                 (student(rina) ⊕ teacher(rina))
Vacuums suck.                                                          (vacuum(X) → suck(X))
Red, round things are big.                                             ((red(X) ∧ round(X)) → big(X))
If something likes the cat and it is not red then the cat chases it.   ((lik cat(X) ∧ ¬red(X)) → chas X(cat))


## 3. Grounding and CNF

In [4]:
def subst(f, c):
    if f[0] == 'atom':
        return atom(' '.join(pred(c) if w == 'X' else w for w in f[1].split()), c if f[2] == 'X' else f[2])
    return (f[0],) + tuple(subst(g, c) for g in f[1:])


def nnf(f, pos=True):
    op = f[0]
    if op == 'atom':
        return f if pos else ('not', f)
    if op == 'not':
        return nnf(f[1], not pos)
    if op == 'imp':
        return nnf(('or', neg(f[1]), f[2]), pos)
    if op == 'iff':
        return nnf(('and', ('imp', f[1], f[2]), ('imp', f[2], f[1])), pos)
    if op == 'xor':
        return nnf(('and', ('or', f[1], f[2]), ('or', neg(f[1]), neg(f[2]))), pos)
    if not pos:
        op = 'or' if op == 'and' else 'and'
    return (op,) + tuple(nnf(g, pos) for g in f[1:])


def key(f):
    return f'{f[1]}({f[2]})'


def cnf(f):
    if f[0] == 'atom':
        return [frozenset([key(f)])]
    if f[0] == 'not':
        return [frozenset(['~' + key(f[1])])]
    if f[0] == 'and':
        return [c for g in f[1:] for c in cnf(g)]
    out = [frozenset()]
    for g in f[1:]:
        out = [a | b for a in out for b in cnf(g)]
    return [c for c in out if not any('~' + l in c for l in c)]


def negate(l):
    return l[1:] if l.startswith('~') else '~' + l

## 4. Forward chaining with negation as failure

In [5]:
def atoms(f):
    return [f] if f[0] == 'atom' else [a for g in f[1:] for a in atoms(g)]


def literals(f):
    if f[0] == 'and':
        return [l for g in f[1:] for l in literals(g)]
    if f[0] == 'atom' or (f[0] == 'not' and f[1][0] == 'atom'):
        return [f]
    raise ValueError('not a literal')


def program(formulas):
    facts, rules = [], []
    for f in formulas:
        body, head = (f[1], f[2]) if f[0] == 'imp' else (('and',), f)
        body, head = literals(body), literals(head)
        if any(h[0] == 'not' for h in head):
            raise ValueError('negative conclusion')
        for h in head:
            if body:
                rules.append(([key(b) for b in body if b[0] == 'atom'], [key(b[1]) for b in body if b[0] == 'not'], key(h)))
            else:
                facts.append(key(h))
    return facts, rules


def strata(rules):
    level = defaultdict(int)
    for _ in range(len(rules) + 1):
        changed = False
        for pos, negs, head in rules:
            l = max([level[b] for b in pos] + [level[b] + 1 for b in negs] + [0])
            if l > level[head]:
                level[head], changed = l, True
        if not changed:
            break
    return level


def forward_chain(facts, rules):
    level = strata(rules)
    known = {f: None for f in facts}
    for k in sorted({level[h] for _, _, h in rules}):
        layer = [i for i, (_, _, h) in enumerate(rules) if level[h] <= k]
        changed = True
        while changed:
            changed = False
            for i in layer:
                pos, negs, head = rules[i]
                if head not in known and all(b in known for b in pos) and not any(b in known for b in negs):
                    known[head] = i
                    changed = True
    return known


def proof(goal, known, rules, steps):
    if goal in known and known[goal] is not None:
        pos, negs, head = rules[known[goal]]
        for b in pos:
            proof(b, known, rules, steps)
        step = (pos + ['not ' + n for n in negs], head)
    else:
        step = ([], goal if goal in known else 'not ' + goal)
    if step not in steps:
        steps.append(step)
    return steps

## 5. DPLL satisfiability

In [6]:
def dpll(clauses):
    clauses = list(set(clauses))
    while True:
        if not clauses:
            return True
        if frozenset() in clauses:
            return False
        unit = next((c for c in clauses if len(c) == 1), None)
        if unit is None:
            break
        l = next(iter(unit))
        clauses = [c - {negate(l)} for c in clauses if l not in c]
    l = next(iter(min(clauses, key=len)))
    return any(dpll([c - {negate(v)} for c in clauses if v not in c]) for v in (l, negate(l)))

## 6. Entailment

In [7]:
def logic_predict(context, statement, world='auto'):
    parser, kb, skipped = Parser(), [], 0
    for s in sentences(context):
        try:
            kb.append(parser.sentence(s))
        except (ValueError, IndexError):
            skipped += 1
    try:
        q = parser.sentence(statement)
    except (ValueError, IndexError):
        return False, 'unparsed', {'kb': kb}
    info = {'kb': kb, 'query': q}
    consts = parser.consts + sorted(o for o in parser.objects if pred(o) not in {pred(c) for c in parser.consts}) + ['new']
    ground = []
    for f in kb:
        if f[0] == 'exists':
            consts.append(f'sk{len(consts)}')
            f = subst(f[1], consts[-1])
        ground.append(f)
    q = ('or',) + tuple(subst(q[1], c) for c in consts) if q[0] == 'exists' else subst(q, 'new')
    ground = [g for f in ground for g in ([subst(f, c) for c in consts] if has_var(f) else [f])]
    try:
        facts, rules = program(ground)
        goals = literals(q)
        if world != 'closed' and (any(g[0] == 'not' for g in goals) or any(negs for _, negs, _ in rules)):
            names = {pred(c) for c in consts}
            words = [a[1].split() for f in ground + [q] for a in atoms(f)]
            if world == 'open' or skipped or not all(len(w) == 1 or ' '.join(w[1:]) in names for w in words):
                raise ValueError('open world')
    except ValueError:
        clauses = [c for f in ground for c in cnf(nnf(f))]
        if not dpll(clauses):
            return False, 'inconsistent', info
        return not dpll(clauses + cnf(nnf(neg(q)))), 'dpll', info
    known = forward_chain(facts, rules)
    info['proof'] = [s for g in goals for s in proof(key(g) if g[0] == 'atom' else key(g[1]), known, rules, [])]
    answer = all((key(g) in known) if g[0] == 'atom' else (key(g[1]) not in known) for g in goals)
    return answer, 'forward chaining', info

## 7. Counterfactuals with a structural causal model

In [8]:
def var(text):
    w = text.split()
    return [x for x in w if x not in ('not', 'occur', 'occurs', 'together')][-1], 'not' not in w


def causal_expr(text):
    return [[var(v) for v in conj.split(' and ')] for conj in text.split(' or ')]


def evaluate(expr, values):
    return any(all(values[v] == s for v, s in conj) for conj in expr)


def causal_predict(context, statement, strict=False):
    t = context.lower().strip()
    eqs = {}
    for piece in re.split(r',\s*(?:and\s+)?', re.match(r'we know that (.+?)(?:\.|$)', t)[1]):
        m = re.match(r'(.+?) causes? (.+)', piece)
        if m:
            cause = causal_expr(m[1])
        for effect in (m[2] if m else piece).split(' and '):
            name, sign = var(effect)
            eqs[name] = (cause, sign)
    obs = re.search(r'we observed (.+?)(?:\.|$)', t)
    obs = [var(v) for v in re.split(r',\s*|\s+and\s+', obs[1].strip())] if obs else []
    q = statement.lower().strip(' ?')
    m = re.match(r'would (\w+) occur if (.+?)(?: instead of (.+))?$', q)
    if m:
        target, do = m[1], m[2]
        if strict and m[3]:
            obs.append(var(m[3]))
    else:
        m = re.match(r'assume (.+?), and based on this assumption, further suppose (.+?)\. would (\w+) occur', q)
        target, do = m[3], m[1] + ' and ' + m[2]
    do = dict(var(v) for v in do.split(' and '))
    names = set(eqs) | {v for c, _ in eqs.values() for conj in c for v, _ in conj} | set(do) | {target}
    roots = sorted(names - set(eqs))

    def solve(u, do):
        values = {**u, **do}
        for _ in range(len(eqs) + 1):
            for v, (cause, sign) in eqs.items():
                if v not in do and all(x in values for conj in cause for x, _ in conj):
                    values[v] = evaluate(cause, values) == sign
        if len(values) < len(names):
            raise ValueError('cyclic graph')
        return values

    worlds = []
    for bits in itertools.product([False, True], repeat=len(roots)):
        u = dict(zip(roots, bits))
        if all(solve(u, {})[v] == s for v, s in obs):
            worlds.append((u, solve(u, do)[target]))
    info = {'equations': eqs, 'observed': obs, 'do': do, 'target': target, 'worlds': worlds}
    return bool(worlds) and all(w[1] for w in worlds), 'causal model', info

## 8. Prediction

In [9]:
def predict(context, statement, strict=False, world='auto'):
    try:
        if re.search(r'\bcauses?\b', str(context)) and re.search(r'\boccur\b', statement):
            return causal_predict(context, statement, strict)
        return logic_predict(context, statement, world)
    except Exception:
        return False, 'error', {}


def explain(context, statement):
    answer, method, info = predict(context, statement)
    print(context, '\nQuery:', statement)
    print(f'Prediction: {answer}  ({method})')
    if method == 'causal model':
        for v, (cause, sign) in info['equations'].items():
            rhs = ' ∨ '.join(' ∧ '.join(('' if s else '¬') + x for x, s in conj) for conj in cause)
            print(f'  {v} := {"" if sign else "¬"}({rhs})')
        print('  observed:', info['observed'], ' do:', info['do'])
        for u, out in info['worlds']:
            print(f'  roots {u} -> {info["target"]} = {out}')
        return
    for f in info.get('kb', []):
        print('  KB:', show(f))
    if 'query' in info:
        print('  Q :', show(info['query']))
    for body, head in info.get('proof', []):
        if body:
            print('  step:', ' ∧ '.join(body), '⇒', head)
        else:
            print('  step:', 'not derivable:' if head.startswith('not ') else 'given', head.replace('not ', ''))
    if method == 'dpll':
        print('  KB ∧ ¬Q is', 'unsatisfiable' if answer else 'satisfiable')


explain('Bob is young. Bob is kind. If someone is young, then they are smart. '
        'If someone is smart and kind, then they are successful.', 'Bob is successful.')
print()
explain('The cat is red. The bear likes the cat. If something likes the cat and it is not red then it is big.',
        'The bear is big.')

Bob is young. Bob is kind. If someone is young, then they are smart. If someone is smart and kind, then they are successful. 
Query: Bob is successful.
Prediction: True  (forward chaining)
  KB: young(bob)
  KB: kind(bob)
  KB: (young(X) → smart(X))
  KB: ((smart(X) ∧ kind(X)) → successful(X))
  Q : successful(bob)
  step: given young(bob)
  step: young(bob) ⇒ smart(bob)
  step: given kind(bob)
  step: smart(bob) ∧ kind(bob) ⇒ successful(bob)

The cat is red. The bear likes the cat. If something likes the cat and it is not red then it is big. 
Query: The bear is big.
Prediction: True  (forward chaining)
  KB: red(cat)
  KB: lik cat(bear)
  KB: ((lik cat(X) ∧ ¬red(X)) → big(X))
  Q : big(bear)
  step: given lik cat(bear)
  step: lik cat(bear) ∧ not red(bear) ⇒ big(bear)


## 9. Evaluation on the public data

In [10]:
def run(rows):
    out = []
    for r in rows:
        start = time.perf_counter()
        label, method, _ = predict(r['context'], r['statement'])
        out.append({**r, 'pred': label, 'method': method, 'seconds': time.perf_counter() - start})
    return out


def summary(results, group):
    groups = defaultdict(list)
    for r in results:
        groups[group(r)].append(r)
    print(f"{'group':38s}{'n':>6s}{'accuracy':>10s}{'all-False':>11s}{'time (s)':>10s}")
    for g, rs in sorted(groups.items(), key=lambda x: str(x[0])):
        acc = 100 * sum(r['pred'] == r['label'] for r in rs) / len(rs)
        base = 100 * sum(not r['label'] for r in rs) / len(rs)
        print(f"{str(g):38s}{len(rs):6d}{acc:10.2f}{base:11.2f}{sum(r['seconds'] for r in rs):10.3f}")


labelled = [r for r in rows if 'label' in r]
start = time.perf_counter()
results = run(labelled)
total = time.perf_counter() - start
if results:
    summary(results, lambda r: f"{r['dataset']} / {r['split']}")
    held_out = [r for r in results if r['split'] != 'train']
    print(f"\nAccuracy on evaluation splits: {100 * sum(r['pred'] == r['label'] for r in held_out) / len(held_out):.2f}% "
          f"({len(held_out)} queries)")
    print(f'Total inference time: {total:.2f} s for {len(results)} queries '
          f'({1000 * total / len(results):.2f} ms per query)')
else:
    print('No labelled public data found; skipping evaluation.')

group                                      n  accuracy  all-False  time (s)
CounterBench / public                   1200     94.17      50.42     0.202
FOLIO / train                           1001     62.84      61.24     0.776
FOLIO / validation                       203     67.98      64.53     0.139
RuleTaker / ood_test                     375     99.47      65.07     0.200
RuleTaker / test                         250     97.60      37.60     0.141
RuleTaker / train                       1000     98.30      40.60     0.460
RuleTaker-standard / synthetic          1000    100.00      50.70     0.621

Accuracy on evaluation splits: 95.28% (3028 queries)
Total inference time: 2.54 s for 5029 queries (0.51 ms per query)


In [11]:
for field, name in [('depth', 'RuleTaker'), ('question_type', 'CounterBench'), ('original_label', 'FOLIO')]:
    part = [r for r in results if r['dataset'] == name and r['split'] != 'train' and field in r]
    if part:
        summary(part, lambda r: f'{field} {r[field]}')
        print()
for k, v in sorted(Counter((r['dataset'], r['method']) for r in results).items()):
    print(k, v)

group                                      n  accuracy  all-False  time (s)
depth 1                                   54    100.00       0.00     0.031
depth 2                                   94     94.68      44.68     0.053
depth 3                                  102     99.02      50.98     0.057
depth 4                                  250    100.00      47.60     0.135
depth 5                                  125     98.40     100.00     0.065

group                                      n  accuracy  all-False  time (s)
question_type backdoor                   200     90.00      52.50     0.044
question_type basic                      250    100.00      50.00     0.032
question_type conditional                250     88.00      50.00     0.053
question_type joint                      250     96.00      50.00     0.036
question_type nested                     250     96.00      50.00     0.037

group                                      n  accuracy  all-False  time (s)
original_l

In [12]:
counter = [r for r in labelled if r['dataset'] == 'CounterBench']
for strict in (False, True):
    if counter:
        acc = sum(predict(r['context'], r['statement'], strict)[0] == r['label'] for r in counter) / len(counter)
        print(f"'instead of' as an extra observation = {strict}: {100 * acc:.2f}%")

'instead of' as an extra observation = False: 94.17%


'instead of' as an extra observation = True: 90.42%


In [13]:
for world in ('open', 'closed', 'auto'):
    line = []
    for name, split in [('RuleTaker-standard', 'synthetic'), ('RuleTaker', 'test'), ('FOLIO', 'validation'), ('FOLIO', 'train')]:
        part = [r for r in labelled if r['dataset'] == name and r['split'] == split]
        if part:
            acc = sum(predict(r['context'], r['statement'], world=world)[0] == r['label'] for r in part) / len(part)
            line.append(f'{name} {split}: {100 * acc:.2f}%')
    print(f'world={world:6s}', ' | '.join(line))

world=open   RuleTaker-standard synthetic: 66.30% | RuleTaker test: 97.60% | FOLIO validation: 67.98% | FOLIO train: 62.94%


world=closed RuleTaker-standard synthetic: 100.00% | RuleTaker test: 97.60% | FOLIO validation: 64.04% | FOLIO train: 59.24%


world=auto   RuleTaker-standard synthetic: 100.00% | RuleTaker test: 97.60% | FOLIO validation: 67.98% | FOLIO train: 62.84%


## 10. Reasoning traces

In [14]:
by_id = {r['id']: r for r in results}
for i in ['ruletaker_ood_test_0190', 'std_ruletaker_0391', 'folio_train_715', 'folio_validation_578', 'counterbench_alpha_V1_209']:
    if i in by_id:
        print('=' * 20, i, '| gold:', by_id[i]['label'])
        explain(by_id[i]['context'], by_id[i]['statement'])
        print()

==================== ruletaker_ood_test_0190 | gold: True
Facts:
The wall travels far.
Ethan does yoga.
James is friendly.
Rules:
Assuming ethan does yoga, we can say the plane is friendly.
If ethan becomes famous, then james travels far.
When the wall becomes famous, we can conclude ethan becomes famous.
Assuming the plane travels far, we can say the wall becomes famous.
If james is friendly, then the plane travels far. 
Query: James travels far.
Prediction: True  (forward chaining)
  KB: travel far(wall)
  KB: yoga(ethan)
  KB: friendly(james)
  KB: (yoga(ethan) → friendly(plane))
  KB: (becom famous(ethan) → travel far(james))
  KB: (becom famous(wall) → becom famous(ethan))
  KB: (travel far(plane) → becom famous(wall))
  KB: (friendly(james) → travel far(plane))
  Q : travel far(james)
  step: given friendly(james)
  step: friendly(james) ⇒ travel far(plane)
  step: travel far(plane) ⇒ becom famous(wall)
  step: becom famous(wall) ⇒ becom famous(ethan)
  step: becom famous(ethan) 

## 11. Submission file

In [15]:
test = load(TEST_FILE) if TEST_FILE else [r for r in rows if r['split'] != 'train']
if not test:
    print('Nothing to predict: set TEST_FILE in the first cell or add the data folder.')
start = time.perf_counter()
predictions = [(r['id'], predict(r['context'], r['statement'])[0]) for r in test]
seconds = time.perf_counter() - start
with open('submission.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'predicted_label'])
    writer.writerows(predictions)
print(f'submission.csv: {len(predictions)} rows, inference time {seconds:.2f} s')
gold = [(p, r['label']) for (_, p), r in zip(predictions, test) if 'label' in r]
if gold:
    print(f'Accuracy on these rows: {100 * sum(p == g for p, g in gold) / len(gold):.2f}% ({len(gold)} labelled)')

submission.csv: 3028 rows, inference time 1.25 s
Accuracy on these rows: 95.28% (3028 labelled)
